# vLLM on Kaggle for the Linux assistant

This notebook launches an OpenAI-compatible multimodal server on Kaggle for the Linux assistant.

Target stack:
- `vLLM`
- `Qwen/Qwen2.5-VL-7B-Instruct`
- `subprocess.Popen(...)` lifecycle
- `/v1/models` readiness probe
- image-aware chat test

It is designed as a temporary remote vision backend that can later be relayed through your Oracle VM.


In [ ]:
# RUN GUARD
RUN_NOTEBOOK = False

if not RUN_NOTEBOOK:
    raise RuntimeError(
        "Notebook execution is disabled. Set RUN_NOTEBOOK = True only after checking the config cell."
    )


## Configuration

Adjust these values only if needed. The defaults target Kaggle T4x2 for `Qwen2.5-VL-7B-Instruct`.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
HOST = "127.0.0.1"
PORT = 8000
TENSOR_PARALLEL_SIZE = 2
DTYPE = "half"
ATTENTION_BACKEND = "TRITON_ATTN"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.92
LIMIT_MM_PER_PROMPT = '{"image":1,"video":0}'
STARTUP_TIMEOUT_SECONDS = 240
CLIENT_TIMEOUT_SECONDS = 120
TEST_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
TUNNEL_HOSTNAME = ""
CLOUDFLARE_TUNNEL_TOKEN = ""
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
CLOUDFLARED_BIN = ""  # optional explicit path; otherwise PATH, then WORK_ROOT/cloudflared
WORK_ROOT = "/kaggle/working/vllm_server"
LOG_FILE = f"{WORK_ROOT}/vllm.log"
PID_FILE = f"{WORK_ROOT}/vllm.pid"
CLOUDFLARED_LOG_FILE = f"{WORK_ROOT}/cloudflared.log"
CLOUDFLARED_PID_FILE = f"{WORK_ROOT}/cloudflared.pid"


: 

## Install packages

Run once per fresh Kaggle session.


In [ ]:
%pip install -U "vllm>=0.8.5" openai httpx requests

## Cloudflare binary resolution


In [ ]:
import shutil
from pathlib import Path

work_root = Path(WORK_ROOT)
work_root.mkdir(parents=True, exist_ok=True)
resolved = CLOUDFLARED_BIN or shutil.which("cloudflared") or str(work_root / "cloudflared")
print({"cloudflared_candidate": resolved})


In [ ]:
# Optional: if you explicitly want a system package instead of PATH/local binary,
# install cloudflared yourself before running the tunnel helpers.
# The helpers below prefer:
# 1. CLOUDFLARED_BIN if set
# 2. cloudflared from PATH
# 3. WORK_ROOT/cloudflared downloaded on demand
print("Using helper-managed cloudflared resolution; no apt step required.")


In [ ]:
print("Skip: cloudflared will be resolved automatically when you start the tunnel.")


## Runtime helpers

This is the actual operational layer: start, stop, restart, logs, port checks, and readiness probing.


In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import time
from typing import Any
from urllib import request, error


WORK_ROOT_PATH = Path(WORK_ROOT)
LOG_PATH = Path(LOG_FILE)
PID_PATH = Path(PID_FILE)
CLOUDFLARED_LOG_PATH = Path(CLOUDFLARED_LOG_FILE)
CLOUDFLARED_PID_PATH = Path(CLOUDFLARED_PID_FILE)
TRYCLOUDFLARE_PATTERN = re.compile(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com")
STARTUP_MARKERS = [
    ("process_started", "process started"),
    ("engine_init", "Initializing a V1 LLM engine"),
    ("workers_ready", "distributed_init_method="),
    ("weights_loading", "Loading safetensors checkpoint shards"),
    ("weights_loaded", "Loading weights took"),
    ("compile_cache", "torch.compile took"),
    ("graph_register", "Registering 112 cuda graph addresses"),
]
ERROR_MARKERS = [
    "RuntimeError:",
    "ValueError:",
    "CalledProcessError",
    "Ninja build failed",
    "cannot find -lcuda",
    "WorkerProc failed to start",
]
WORK_ROOT_PATH.mkdir(parents=True, exist_ok=True)


def is_port_open(host: str = HOST, port: int = PORT, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def read_pid() -> int | None:
    if not PID_PATH.exists():
        return None
    try:
        return int(PID_PATH.read_text(encoding="utf-8").strip())
    except Exception:
        return None


def read_cloudflared_pid() -> int | None:
    if not CLOUDFLARED_PID_PATH.exists():
        return None
    try:
        return int(CLOUDFLARED_PID_PATH.read_text(encoding="utf-8").strip())
    except Exception:
        return None


def process_exists(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def tail_log(line_count: int = 120) -> str:
    if not LOG_PATH.exists():
        return ""
    lines = LOG_PATH.read_text(encoding="utf-8", errors="ignore").splitlines()
    return "\n".join(lines[-line_count:])


def tail_cloudflared_log(line_count: int = 120) -> str:
    if not CLOUDFLARED_LOG_PATH.exists():
        return ""
    lines = CLOUDFLARED_LOG_PATH.read_text(encoding="utf-8", errors="ignore").splitlines()
    return "\n".join(lines[-line_count:])


def read_log_lines() -> list[str]:
    if not LOG_PATH.exists():
        return []
    return LOG_PATH.read_text(encoding="utf-8", errors="ignore").splitlines()


def summarize_startup_progress(lines: list[str]) -> dict[str, Any]:
    matched: list[str] = []
    last_stage = "waiting for first log line"
    joined = "\n".join(lines)

    if lines:
        last_stage = "process started"

    for stage_id, marker in STARTUP_MARKERS[1:]:
        if marker in joined:
            matched.append(stage_id)
            last_stage = stage_id.replace("_", " ")

    errors = [marker for marker in ERROR_MARKERS if marker in joined]
    return {
        "last_stage": last_stage,
        "matched_stages": matched,
        "errors": errors,
    }


def download_cloudflared() -> Path:
    local_bin = WORK_ROOT_PATH / "cloudflared"
    if local_bin.exists():
        return local_bin
    request.urlretrieve(CLOUDFLARED_URL, local_bin)
    local_bin.chmod(0o755)
    return local_bin


def resolve_cloudflared_bin() -> Path:
    if CLOUDFLARED_BIN:
        explicit = Path(CLOUDFLARED_BIN)
        if explicit.exists():
            return explicit
        raise FileNotFoundError(f"CLOUDFLARED_BIN does not exist: {explicit}")

    system_bin = shutil.which("cloudflared")
    if system_bin:
        return Path(system_bin)

    return download_cloudflared()


def build_server_command() -> list[str]:
    return [
        "vllm",
        "serve",
        MODEL_ID,
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--tensor-parallel-size",
        str(TENSOR_PARALLEL_SIZE),
        "--dtype",
        DTYPE,
        "--attention-backend",
        ATTENTION_BACKEND,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--limit-mm-per-prompt",
        LIMIT_MM_PER_PROMPT,
    ]


def build_cloudflared_command() -> list[str]:
    binary = resolve_cloudflared_bin()
    if not CLOUDFLARE_TUNNEL_TOKEN:
        return [str(binary), "tunnel", "--url", f"http://localhost:{PORT}"]
    return [str(binary), "tunnel", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN]

def start_server() -> subprocess.Popen[str]:
    existing_pid = read_pid()
    if existing_pid and process_exists(existing_pid):
        raise RuntimeError(f"Server already running with pid {existing_pid}")

    log_handle = open(LOG_PATH, "w", encoding="utf-8")
    process = subprocess.Popen(
        build_server_command(),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,
        text=True,
    )
    PID_PATH.write_text(str(process.pid), encoding="utf-8")
    return process


def stop_server() -> None:
    pid = read_pid()
    if pid is None:
        print("No pid file; nothing to stop.")
        return

    if not process_exists(pid):
        PID_PATH.unlink(missing_ok=True)
        print("Pid file removed; process already gone.")
        return

    os.killpg(os.getpgid(pid), signal.SIGTERM)
    deadline = time.time() + 20
    while time.time() < deadline:
        if not process_exists(pid):
            PID_PATH.unlink(missing_ok=True)
            print("Server stopped cleanly.")
            return
        time.sleep(0.5)

    os.killpg(os.getpgid(pid), signal.SIGKILL)
    PID_PATH.unlink(missing_ok=True)
    print("Server force-killed.")


def start_cloudflared() -> subprocess.Popen[str]:
    existing_pid = read_cloudflared_pid()
    if existing_pid and process_exists(existing_pid):
        raise RuntimeError(f"cloudflared already running with pid {existing_pid}")

    log_handle = open(CLOUDFLARED_LOG_PATH, "w", encoding="utf-8")
    process = subprocess.Popen(
        build_cloudflared_command(),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,
        text=True,
    )
    CLOUDFLARED_PID_PATH.write_text(str(process.pid), encoding="utf-8")
    return process


def stop_cloudflared() -> None:
    pid = read_cloudflared_pid()
    if pid is None:
        print("No cloudflared pid file; nothing to stop.")
        return

    if not process_exists(pid):
        CLOUDFLARED_PID_PATH.unlink(missing_ok=True)
        print("cloudflared pid file removed; process already gone.")
        return

    os.killpg(os.getpgid(pid), signal.SIGTERM)
    deadline = time.time() + 20
    while time.time() < deadline:
        if not process_exists(pid):
            CLOUDFLARED_PID_PATH.unlink(missing_ok=True)
            print("cloudflared stopped cleanly.")
            return
        time.sleep(0.5)

    os.killpg(os.getpgid(pid), signal.SIGKILL)
    CLOUDFLARED_PID_PATH.unlink(missing_ok=True)
    print("cloudflared force-killed.")


def restart_server() -> subprocess.Popen[str]:
    stop_server()
    return start_server()


def restart_cloudflared() -> subprocess.Popen[str]:
    stop_cloudflared()
    return start_cloudflared()


def server_models() -> dict[str, Any] | None:
    try:
        with request.urlopen(f"http://{HOST}:{PORT}/v1/models", timeout=5) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None


def wait_for_server(timeout_s: int = STARTUP_TIMEOUT_SECONDS) -> bool:
    deadline = time.time() + timeout_s
    last_reported_stage = None
    last_reported_errors: tuple[str, ...] = ()
    while time.time() < deadline:
        models = server_models()
        if is_port_open(HOST, PORT) and models is not None:
            return True

        progress = summarize_startup_progress(read_log_lines())
        stage = progress["last_stage"]
        errors = tuple(progress["errors"])
        if stage != last_reported_stage or errors != last_reported_errors:
            print(json.dumps({"ready": False, "stage": stage, "errors": list(errors)}, ensure_ascii=False))
            last_reported_stage = stage
            last_reported_errors = errors
        time.sleep(2)
    return False


def health_summary() -> dict[str, Any]:
    models = server_models()
    progress = summarize_startup_progress(read_log_lines())
    return {
        "ready": models is not None,
        "startup_stage": progress["last_stage"],
        "startup_errors": progress["errors"],
        "pid": read_pid(),
        "port_open": is_port_open(HOST, PORT),
        "models": models,
        "log_tail": tail_log(80),
    }


def start_server_and_wait(timeout_s: int = STARTUP_TIMEOUT_SECONDS) -> dict[str, Any]:
    process = start_server()
    ready = wait_for_server(timeout_s)
    summary = health_summary()
    summary["started_pid"] = process.pid
    summary["ready"] = ready and summary["models"] is not None
    return summary


def discover_tunnel_url(log_text: str) -> str:
    if TUNNEL_HOSTNAME:
        return f"https://{TUNNEL_HOSTNAME}"
    match = TRYCLOUDFLARE_PATTERN.search(log_text)
    return match.group(0) if match else ""


def tunnel_summary() -> dict[str, Any]:
    pid = read_cloudflared_pid()
    log_tail = tail_cloudflared_log(80)
    public_url = discover_tunnel_url(log_tail)
    return {
        "pid": pid,
        "mode": "token" if CLOUDFLARE_TUNNEL_TOKEN else "quick",
        "hostname": TUNNEL_HOSTNAME or public_url.removeprefix("https://"),
        "public_url": public_url,
        "cloudflared_bin": str(resolve_cloudflared_bin()),
        "running": bool(pid and process_exists(pid)),
        "log_tail": log_tail,
    }


## Launch server

Run this cell to start `vLLM` under notebook control.


In [ ]:
summary = start_server_and_wait()
print(json.dumps({k: v for k, v in summary.items() if k != 'log_tail'}, indent=2, ensure_ascii=False))


## Inspect logs


In [ ]:
print(tail_log(120))


## OpenAI-compatible multimodal test

Use a remote image URL first. After this works, the assistant can send screenshots as `data:image/...;base64,...`.


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url=f"http://{HOST}:{PORT}/v1",
    timeout=CLIENT_TIMEOUT_SECONDS,
)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "describe this image briefly"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": TEST_IMAGE_URL
                    }
                },
            ],
        }
    ],
    max_completion_tokens=200,
)

print(response.choices[0].message.content)


## Optional local data URL test

Use this if you want to verify base64 image input directly before wiring the assistant.


In [ ]:
import base64
from pathlib import Path

# Set this only if you uploaded an image file into the Kaggle session.
LOCAL_TEST_IMAGE = ""

if LOCAL_TEST_IMAGE:
    image_bytes = Path(LOCAL_TEST_IMAGE).read_bytes()
    image_base64 = base64.b64encode(image_bytes).decode("ascii")
    data_url = f"data:image/png;base64,{image_base64}"

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "what do you see here?"},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ],
        max_completion_tokens=200,
    )
    print(response.choices[0].message.content)
else:
    print("Set LOCAL_TEST_IMAGE to run this cell.")


## Stop server


In [ ]:
stop_server()


## Oracle relay notes

Your next step is to expose this `/v1` surface through your Oracle VM, then point the assistant at the Oracle URL instead of Kaggle directly.

What the assistant will need later:

```env
VISION_PROVIDER=openai_compat
OPENAI_COMPAT_BASE_URL=https://YOUR-ORACLE-ENDPOINT/v1
OPENAI_COMPAT_API_KEY=EMPTY
OPENAI_COMPAT_MODEL=Qwen/Qwen2.5-VL-7B-Instruct
```


## Cloudflare Tunnel helpers

Use these cells if you want to expose the Kaggle `vLLM` endpoint directly through Cloudflare instead of relaying through Oracle.


In [ ]:
tunnel_process = start_cloudflared()
print(json.dumps({k: v for k, v in tunnel_summary().items() if k != 'log_tail'}, indent=2, ensure_ascii=False))


## Inspect Cloudflare Tunnel logs


In [ ]:
print(tail_cloudflared_log(120))


## Stop Cloudflare Tunnel


In [ ]:
stop_cloudflared()
